# Assignment 1 — Myanmar POS Tagging with CRF

**Student:** Moe Pyae Hein  
**Dataset:** myPOS Version 3.0  
**Model:** Conditional Random Field (CRF)

## Objective

This notebook builds a simple Myanmar POS tagger.

```text
Word-segmented sentence
→ Simple word features
→ CRF
→ POS tags
```

Example:

```text
ကျွန်တော်/pron က/ppm စာ/n လေ့လာ/v တယ်/ppm
```


## 1. Install packages

Run this cell once.


In [1]:
%pip install -q sklearn-crfsuite scikit-learn pandas

Note: you may need to restart the kernel to use updated packages.


## 2. Import libraries

In [2]:
from pathlib import Path
from collections import Counter

import pandas as pd
import sklearn_crfsuite

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn_crfsuite import metrics

RANDOM_SEED = 42

print("Libraries imported successfully.")


Libraries imported successfully.


## 3. Set the myPOS 3.0 corpus path

In [3]:
PROJECT_DIR = Path.cwd()

CORPUS_FILE = (
    PROJECT_DIR
    / "data"
    / "myPOS"
    / "corpus-ver-3.0"
    / "corpus"
    / "mypos-ver.3.0.txt"
)

print("Corpus file:", CORPUS_FILE)
print("File exists:", CORPUS_FILE.exists())

if not CORPUS_FILE.exists():
    raise FileNotFoundError(
        "myPOS 3.0 corpus was not found. Check data/myPOS."
    )


Corpus file: /mnt/d/AI-Engineering-Batch2/Moe-AIEF-Learning/assignments/batch2-assignment-1/data/myPOS/corpus-ver-3.0/corpus/mypos-ver.3.0.txt
File exists: True


## 4. Parse one tagged sentence

Each corpus item normally has this format:

```text
word/POS
```

Some compound words contain `|`.


In [4]:
def parse_sentence(line):
    sentence = []

    for item in line.strip().split():
        for piece in item.split("|"):
            if "/" not in piece:
                continue

            word, tag = piece.rsplit("/", 1)
            word = word.strip()
            tag = tag.strip()

            if word and tag:
                sentence.append((word, tag))

    return sentence


sample_line = "ကျွန်တော်/pron က/ppm စာ/n လေ့လာ/v တယ်/ppm ။/punc"
print(parse_sentence(sample_line))


[('ကျွန်တော်', 'pron'), ('က', 'ppm'), ('စာ', 'n'), ('လေ့လာ', 'v'), ('တယ်', 'ppm'), ('။', 'punc')]


## 5. Load the full corpus

Each non-empty line is treated as one tagged sequence.


In [5]:
sentences = []

with CORPUS_FILE.open(
    "r",
    encoding="utf-8",
    errors="replace",
) as file:
    for line in file:
        line = line.strip()

        if not line:
            continue

        sentence = parse_sentence(line)

        if sentence:
            sentences.append(sentence)

print("Loaded sequences:", len(sentences))
print("First sequence:")
print(sentences[0])


Loaded sequences: 43196
First sequence:
[('ဒီ', 'adj'), ('ဆေး', 'n'), ('က', 'ppm'), ('၁၀၀', 'num'), ('ရာခိုင်နှုန်း', 'n'), ('ဆေးဘက်ဝင်', 'adj'), ('အပင်', 'n'), ('များ', 'part'), ('မှ', 'ppm'), ('ဖော်စပ်', 'v'), ('ထား', 'part'), ('တာ', 'part'), ('ဖြစ်', 'v'), ('တယ်', 'ppm'), ('။', 'punc')]


## 6. Inspect the dataset

In [6]:
all_words = [
    word
    for sentence in sentences
    for word, tag in sentence
]

all_tags = [
    tag
    for sentence in sentences
    for word, tag in sentence
]

word_counts = Counter(all_words)
tag_counts = Counter(all_tags)

print("Total word tokens:", len(all_words))
print("Unique words:", len(word_counts))
print("Number of POS tags:", len(tag_counts))

tag_table = pd.DataFrame(
    tag_counts.most_common(),
    columns=["POS tag", "Count"],
)

display(tag_table)


Total word tokens: 564517
Unique words: 24831
Number of POS tags: 15


,POS tag,Count
0,part,135267
1,n,122892
2,ppm,86490
3,v,84080
4,punc,54108
5,pron,20413
6,conj,17808
7,adj,16430
8,adv,10711
9,num,5942


## 7. Split training and testing data

- 80% training
- 20% testing
- fixed random seed for reproducibility


In [7]:
train_sentences, test_sentences = train_test_split(
    sentences,
    test_size=0.20,
    random_state=RANDOM_SEED,
    shuffle=True,
)

print("Training sequences:", len(train_sentences))
print("Testing sequences:", len(test_sentences))


Training sequences: 34556
Testing sequences: 8640


## 8. Baseline features

The baseline uses:

- current word
- previous word
- next word
- beginning and end of sentence


In [8]:
def baseline_features(sentence, index):
    word = sentence[index][0]

    features = {
        "word": word,
    }

    if index == 0:
        features["BOS"] = True
    else:
        features["previous_word"] = sentence[index - 1][0]

    if index == len(sentence) - 1:
        features["EOS"] = True
    else:
        features["next_word"] = sentence[index + 1][0]

    return features


## 9. Improved features

The improved model adds only:

- word length
- first character
- last character
- digit detection
- punctuation detection


In [9]:
def improved_features(sentence, index):
    word = sentence[index][0]

    features = {
        "word": word,
        "word_length": len(word),
        "first_character": word[0],
        "last_character": word[-1],
        "contains_digit": any(
            character.isdigit()
            for character in word
        ),
        "is_punctuation": word in {
            "။", "၊", ".", ",", "!", "?", "(", ")"
        },
    }

    if index == 0:
        features["BOS"] = True
    else:
        features["previous_word"] = sentence[index - 1][0]

    if index == len(sentence) - 1:
        features["EOS"] = True
    else:
        features["next_word"] = sentence[index + 1][0]

    return features


## 10. Convert sentences into CRF input

- `X` contains feature dictionaries.
- `y` contains correct POS tags.


In [10]:
def sentence_to_features(sentence, feature_function):
    return [
        feature_function(sentence, index)
        for index in range(len(sentence))
    ]


def sentence_to_labels(sentence):
    return [
        tag
        for word, tag in sentence
    ]


X_train_baseline = [
    sentence_to_features(sentence, baseline_features)
    for sentence in train_sentences
]

X_test_baseline = [
    sentence_to_features(sentence, baseline_features)
    for sentence in test_sentences
]

X_train_improved = [
    sentence_to_features(sentence, improved_features)
    for sentence in train_sentences
]

X_test_improved = [
    sentence_to_features(sentence, improved_features)
    for sentence in test_sentences
]

y_train = [
    sentence_to_labels(sentence)
    for sentence in train_sentences
]

y_test = [
    sentence_to_labels(sentence)
    for sentence in test_sentences
]

print("Prepared training sequences:", len(X_train_improved))
print("Prepared testing sequences:", len(X_test_improved))
print("\nExample feature dictionary:")
print(X_train_improved[0][0])


Prepared training sequences: 34556
Prepared testing sequences: 8640

Example feature dictionary:
{'word': 'အဲဒါ', 'word_length': 4, 'first_character': 'အ', 'last_character': 'ါ', 'contains_digit': False, 'is_punctuation': False, 'BOS': True, 'next_word': 'နောက်'}


## 11. Train the baseline CRF

In [11]:
baseline_crf = sklearn_crfsuite.CRF(
    algorithm="lbfgs",
    c1=0.1,
    c2=0.1,
    max_iterations=100,
    all_possible_transitions=True,
)

baseline_crf.fit(
    X_train_baseline,
    y_train,
)

baseline_predictions = baseline_crf.predict(
    X_test_baseline
)

baseline_accuracy = metrics.flat_accuracy_score(
    y_test,
    baseline_predictions,
)

print(f"Baseline token accuracy: {baseline_accuracy:.4f}")


Baseline token accuracy: 0.9557


## 12. Train the improved CRF

In [12]:
improved_crf = sklearn_crfsuite.CRF(
    algorithm="lbfgs",
    c1=0.1,
    c2=0.1,
    max_iterations=100,
    all_possible_transitions=True,
)

improved_crf.fit(
    X_train_improved,
    y_train,
)

improved_predictions = improved_crf.predict(
    X_test_improved
)

improved_accuracy = metrics.flat_accuracy_score(
    y_test,
    improved_predictions,
)

print(f"Improved token accuracy: {improved_accuracy:.4f}")
print(
    f"Accuracy difference: "
    f"{improved_accuracy - baseline_accuracy:+.4f}"
)


Improved token accuracy: 0.9597
Accuracy difference: +0.0040


## 13. Compare the models

In [13]:
comparison_table = pd.DataFrame(
    {
        "Model": ["Baseline CRF", "Improved CRF"],
        "Token accuracy": [
            baseline_accuracy,
            improved_accuracy,
        ],
    }
)

display(comparison_table)


,Model,Token accuracy
0,Baseline CRF,0.955654
1,Improved CRF,0.959667


## 14. Evaluate the improved model

In [14]:
flat_y_test = [
    tag
    for sentence_tags in y_test
    for tag in sentence_tags
]

flat_y_pred = [
    tag
    for sentence_tags in improved_predictions
    for tag in sentence_tags
]

labels = sorted(set(flat_y_test))

print(
    classification_report(
        flat_y_test,
        flat_y_pred,
        labels=labels,
        digits=4,
        zero_division=0,
    )
)


              precision    recall  f1-score   support

         abb     0.9815    0.7260    0.8346        73
         adj     0.8542    0.8004    0.8264      3316
         adv     0.9024    0.8298    0.8646      2150
        conj     0.8992    0.9236    0.9112      3624
          fw     0.9829    0.9845    0.9837       644
         int     0.9412    0.8175    0.8750       137
           n     0.9526    0.9717    0.9620     24736
         num     0.9983    0.9889    0.9936      1174
        part     0.9664    0.9676    0.9670     26919
         ppm     0.9783    0.9841    0.9812     17509
        pron     0.9645    0.9583    0.9614      4056
        punc     0.9992    0.9999    0.9995     10859
          sb     1.0000    0.8222    0.9024        45
          tn     0.9704    0.9456    0.9578      1177
           v     0.9486    0.9347    0.9416     16963

    accuracy                         0.9597    113382
   macro avg     0.9560    0.9103    0.9308    113382
weighted avg     0.9594   

## 15. Show the first 20 errors

In [15]:
errors = []

for sentence, gold_tags, predicted_tags in zip(
    test_sentences,
    y_test,
    improved_predictions,
):
    for (word, gold_tag), predicted_tag in zip(
        sentence,
        predicted_tags,
    ):
        if gold_tag != predicted_tag:
            errors.append(
                {
                    "word": word,
                    "gold_tag": gold_tag,
                    "predicted_tag": predicted_tag,
                }
            )

        if len(errors) >= 20:
            break

    if len(errors) >= 20:
        break

display(pd.DataFrame(errors))


,word,gold_tag,predicted_tag
0,ဘယ်လို,adj,adv
1,ကောင်း,v,adj
2,ကောင်း,adj,v
3,ပုံသေ,adj,adv
4,ထူးခြား,adv,adj
5,ကြည့်,part,v
6,ယုံ,v,part
7,အတွက်အချက်,n,adv
8,မြန်,v,adv
9,ယုံ,part,v


## 16. Test a custom sentence

The sentence must already be separated into words.


In [16]:
def predict_pos(words):
    temporary_sentence = [
        (word, "")
        for word in words
    ]

    features = sentence_to_features(
        temporary_sentence,
        improved_features,
    )

    predicted_tags = improved_crf.predict_single(
        features
    )

    return list(zip(words, predicted_tags))


custom_sentence = [
    "ကျွန်တော်",
    "က",
    "AI",
    "ကို",
    "လေ့လာ",
    "နေ",
    "တယ်",
    "။",
]

predict_pos(custom_sentence)


[('ကျွန်တော်', 'pron'),
 ('က', 'ppm'),
 ('AI', 'fw'),
 ('ကို', 'ppm'),
 ('လေ့လာ', 'v'),
 ('နေ', 'part'),
 ('တယ်', 'ppm'),
 ('။', 'punc')]

## 17. Conclusion

Fill in the values after running the notebook.

### Results

- Loaded sequences: **[43196]**
- Total tokens: **[564517]**
- Baseline accuracy: **[0.9557]**
- Improved accuracy: **[0.9597]**
- Macro F1-score: **[0.9308]**
- Weighted F1-score: **[0.9594]**

### Discussion

The baseline model used the current word and neighboring words.

The improved model added word length, first character,
last character, digit detection, and punctuation detection.

Explain whether the additional features improved accuracy.

### Main errors

Write two or three examples from the error table.

### Limitations

- Input must already be word-segmented.
- Some words have different POS tags in different contexts.
- Rare POS tags have fewer training examples.
- Only one train/test split was used.
- Hyperparameters were not fully tuned.

### Final statement

This experiment shows that CRF can use word information,
neighboring words, and simple character features for Myanmar
POS tagging.


## References

1. https://github.com/ye-kyaw-thu/myPOS  
2. https://github.com/ye-kyaw-thu/AIE-F-B2/tree/main/notebooks  
3. https://sklearn-crfsuite.readthedocs.io/
